# 2.3 Align Qwen3-0.6B with an RL pass (RLAIF / RLVR + DPO) on Amazon SageMaker AI

This notebook shows how to:
1. Fine-tune `Qwen/Qwen3-0.6B` with QLoRA on 200 medical reasoning examples, with the same script and hyperparameters as Lab 02.01 (or skip fine-tuning and start RL from the base model).
2. Define a reward signal that captures a quality dimension the ROUGE benchmark cannot score, using a verifiable reward (RLVR), an Amazon Bedrock judge (RLAIF), or both.
3. Configure and launch an RL training job on SageMaker AI Training Jobs that runs DPO with a KL penalty on top of the starting policy.
4. Re-evaluate the models on the same benchmark, compare the metrics, and test whether the gain is statistically significant.
5. Deploy the RL-tuned model to a SageMaker AI real-time endpoint, which `task_04_responsible_ai/04.01_bedrock_guardrails_apply_guardrail_api.ipynb` protects with Amazon Bedrock Guardrails.

Uses the **SageMaker Python SDK v3** (`sagemaker.core`) throughout.

> **Instances used:** `ml.g5.2xlarge` (1x A10G, 24 GB VRAM) for fine-tuning and `ml.g5.12xlarge` (4x A10G) for the RL pass, one training job at a time. Both jobs use a PyTorch training container with a SOCI index, so the instance starts the job before the full image has downloaded. The lab runs a fine-tuning job of about 10 minutes and an RL job of about 20 minutes, one after the other.

This lab uses **Qwen3-0.6B** instead of the Qwen3-4B model in 02.01 and 02.02, so the whole lab fits a workshop session. The RL pass samples and scores hundreds of answers, and a 0.6B model generates them several times faster. The lab fine-tunes its own model, so it does not depend on 02.01.

ROUGE, the benchmark from task 03, counts word overlap with a reference answer. It cannot tell whether an answer is clinically safe, whether the reasoning supports the conclusion, or whether the model padded its output to overlap with more words. This lab targets those dimensions and then checks that the improvement is real.

| Section | What you do |
|---|---|
| 0 | Prerequisites and setup: session, datasets, and QLoRA fine-tuning |
| 1 | Define a reward signal that captures a quality dimension the benchmark cannot score |
| 2 | Configure the RL training job (DPO with a KL penalty) |
| 3 | Launch the RL training job |
| 4 | Re-evaluate the base, fine-tuned and RL-tuned models on the same benchmark |
| 5 | Compare metrics: baseline, fine-tuned, RL-tuned |
| 6 | Analyze whether the gain is statistically significant |
| 7 | Deploy the RL-tuned model to a real-time endpoint |

**Key takeaway:** you can measure whether RL alignment improved the model on dimensions the benchmark alone missed, with a clear before and after comparison.

All the code for this lab lives in `scripts_rl/`: the job scripts, including the QLoRA script `train.py`, the helpers this notebook imports, and the notebook's own package list. The pipeline in `task_05_fmops/05.02_rl_improvement_pipeline.ipynb` reuses the same scripts.

---

## Use case and RL technique at a glance

### Use case

The model is a medical reasoning assistant: `Qwen/Qwen3-0.6B`, optionally fine-tuned on [medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT). The benchmark from task 03, ROUGE, only measures word overlap with a reference answer. It cannot tell whether an answer is:

* **correct**: the conclusion matches the reference diagnosis or recommendation,
* **safe**: it avoids overconfident or harmful advice and invented doses, and says when a clinician is needed,
* **well reasoned**: the steps support the conclusion, without padding or repetition.

The RL pass optimizes those three qualities directly, and the evaluation then checks that ROUGE did not drop.

### Technique: one round of on-policy DPO with an RLAIF + RLVR reward

The model generates its own answers, a reward signal ranks them, and Direct Preference Optimization (DPO) trains the model to prefer its better answers over its worse ones. A KL penalty keeps it close to the model it started from. All of this runs in one SageMaker AI training job (`scripts_rl/dpo_train.py`):

| Stage | What happens | Settings in this lab |
|---|---|---|
| 1. Generate (on-policy) | The starting model writes several answers to each RL prompt. The prompts are split across the instance's GPUs, one generation process per GPU. | 256 prompts, 4 answers each, temperature 0.9, top_p 0.95, up to 512 tokens, 4 GPUs on `ml.g5.12xlarge` |
| 2. Score | Every answer gets a hybrid reward (section 1): an Amazon Bedrock judge (RLAIF) and deterministic checks (RLVR). | Amazon Nova Pro judge, weighted 0.7; verifiable reward, weighted 0.3 |
| 3. Build pairs | For each prompt, the highest-scoring answer becomes *chosen* and the lowest-scoring one becomes *rejected*. | Pairs kept only when the reward gap is at least 0.05 |
| 4. Train with DPO | DPO trains a 4-bit QLoRA adapter on the pairs. The frozen starting model is the reference policy, and `beta` is the KL penalty. | sigmoid loss, beta 0.1, LoRA r 16 and alpha 32, learning rate 2e-5, 2 epochs, one GPU |
| 5. Merge and evaluate | The adapter is merged into the weights, and the job evaluates the models on the held-out set. | Independent Claude Sonnet 4.5 judge, 100 prompts |

DPO optimizes the same objective as KL-regularized RLHF, `max E[reward] - beta * KL(policy || reference)`, without training a separate reward model and without a PPO rollout loop.

**What it is not.** This is not online RL such as PPO or GRPO: the model does not generate new answers during training, and the lab runs a single round rather than an iterative loop. The reward is not a learned reward model; it comes from an LLM judge plus rules. Training is not distributed across GPUs; only the generation in stage 1 uses all of them.

**Two defenses against reward hacking.** The reward penalizes long and repetitive answers, and the judge is told not to reward length. Section 4 also grades with a judge from a different model family than the training judge, so the RL model is not graded by the model it learned to please.

## 0. Prerequisites

* This lab does not need Lab 02.01. Section 0.4 fine-tunes Qwen3-0.6B with the 02.01 script and hyperparameters on 200 training examples.
* Your SageMaker execution role can call `bedrock:InvokeModel` on the two judge models, and model access is enabled for them in the Amazon Bedrock console. The training jobs use the same role.
* You have quota for one `ml.g5.2xlarge` and one `ml.g5.12xlarge` training job, and for one `ml.g5.2xlarge` endpoint.

Install the required packages. Restart the kernel after this cell completes.

In [ ]:
%pip install -r ./scripts_rl/notebook_requirements.txt --upgrade

## This cell will restart the kernel. Click "OK".

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

***
### 0.1 Setup configuration

In [ ]:
import json
import os
import sys

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from sagemaker.core.helper.session_helper import Session, get_execution_role

sys.path.insert(0, os.path.abspath("./scripts_rl"))  # job code and notebook helpers live in one folder
import rewards
import rl_stats

sagemaker_session = Session()
s3_client = boto3.client('s3')
sm_client = boto3.client('sagemaker')

region = sagemaker_session.boto_session.region_name
role = get_execution_role(sagemaker_session, use_default=True)
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

if default_prefix:
    s3_root = f"s3://{bucket_name}/{default_prefix}"
else:
    s3_root = f"s3://{bucket_name}"

model_id = "Qwen/Qwen3-0.6B"
judges = rewards.default_judge_model_ids(region)

print(f"Region      : {region}")
print(f"S3 root     : {s3_root}")
print(f"Train judge : {judges['train_judge']}  (produces the RL reward)")
print(f"Eval judge  : {judges['eval_judge']}  (independent grader for section 4)")

#### Training container with a SOCI index

Both training jobs use the AWS Deep Learning Container `pytorch-training:2.7-gpu-py312-cu128-ubuntu22.04-sagemaker-v1-soci`. The `-soci` tag carries a [SOCI](https://github.com/awslabs/soci-snapshotter) (Seekable OCI) index, which lets SageMaker AI start the container while the image layers still stream in, instead of waiting for all 12 GB to download first. The image itself is the same build as the plain `2.7-gpu-py312-cu128-ubuntu22.04-sagemaker-v1` tag.

SOCI falls back to a normal full download without any error when the index is missing. The cell below therefore checks in Amazon ECR that the index exists in your Region, and uses the plain tag if it does not.

In [ ]:
DLC_ACCOUNT = "763104351884"  # AWS Deep Learning Containers registry
PYTORCH_TAG = "2.7-gpu-py312-cu128-ubuntu22.04-sagemaker-v1"
ecr_client = boto3.client("ecr", region_name=region)


def has_soci_index(tag):
    try:
        response = ecr_client.batch_get_image(
            registryId=DLC_ACCOUNT,
            repositoryName="pytorch-training",
            imageIds=[{"imageTag": tag}],
            acceptedMediaTypes=["application/vnd.oci.image.index.v1+json"],
        )
    except ClientError as err:
        print(f"Could not inspect {tag}: {err.response['Error']['Code']}")
        return False
    if not response["images"]:
        return False
    manifest = json.loads(response["images"][0]["imageManifest"])
    return any("soci" in (m.get("artifactType") or "") for m in manifest.get("manifests", []))


soci_tag = f"{PYTORCH_TAG}-soci"
use_soci = has_soci_index(soci_tag)
TRAINING_IMAGE = f"{DLC_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/pytorch-training:{soci_tag if use_soci else PYTORCH_TAG}"

print(f"SOCI index found: {use_soci}")
print(f"Training image  : {TRAINING_IMAGE}")

### 0.2 MLflow experiment tracking

Every job in this lab logs to one MLflow experiment, `Qwen3-0.6B-rl-alignment`, on a SageMaker AI MLflow tracking server:

| Run | Logged by | What it records |
|---|---|---|
| `Fine-tuning-<time>` | `scripts_rl/train.py` (section 0.4) | QLoRA hyperparameters, training and validation loss, the datasets, system metrics |
| `rl-dpo-<time>` | `scripts_rl/dpo_train.py` (section 3) | Reward configuration, RL and DPO hyperparameters, preference statistics, the DPO curves (loss, preference margins and accuracies, chosen and rejected rewards), then the evaluation of section 4: mean metrics per model, the paired significance tests and the gate values |

The cell looks up the workshop's tracking server, `genai-mlflow-tracker`, and falls back to any active tracking server in your account. Without a tracking server the jobs still run, without tracking. The SageMaker execution role needs access to the tracking server (`sagemaker-mlflow:*` actions); `AmazonSageMakerFullAccess` includes them.

In [ ]:
from botocore.exceptions import ClientError

MLFLOW_TRACKING_SERVER_NAME = "genai-mlflow-tracker"  # the workshop's tracking server; change it to use your own


def find_tracking_server(name):
    """Return (name, ARN) of the named MLflow tracking server, or of the first active one in the account."""
    try:
        return name, sm_client.describe_mlflow_tracking_server(TrackingServerName=name)["TrackingServerArn"]
    except ClientError:
        pass
    for s in sm_client.list_mlflow_tracking_servers()["TrackingServerSummaries"]:
        if s.get("IsActive") == "Active" and s["TrackingServerStatus"] in ("Created", "Updated"):
            print(f"'{name}' not found, using the active tracking server '{s['TrackingServerName']}'")
            return s["TrackingServerName"], s["TrackingServerArn"]
    return None, ""


mlflow_tracking_server_name, mlflow_tracking_server_uri = find_tracking_server(MLFLOW_TRACKING_SERVER_NAME)
mlflow_experiment_name = "Qwen3-0.6B-rl-alignment"

if mlflow_tracking_server_uri:
    print(f"MLflow tracking server: {mlflow_tracking_server_name}")
    print(f"MLflow experiment     : {mlflow_experiment_name}")
else:
    print("No MLflow tracking server found. The jobs will run without experiment tracking.")

### 0.3 Prepare and upload the datasets

Build two datasets from [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT), the same dataset used in Lab 02.01. Both are disjoint from the 220 rows that the fine-tuning in section 0.4 uses:

* **RL prompts** (256 rows): the RL job samples 4 answers to each prompt and turns their rewards into preference pairs.
* **Held-out evaluation set** (100 rows): every model is scored on exactly these prompts in section 4.

The task 03 benchmark scored 10 samples, which is too few to detect a difference of a few points. 100 paired prompts can detect a gain of about 0.025 to 0.03 on a 0 to 1 score.

Records use the same `messages` format and system prompt as 02.01. The `reference_answer` field holds the final response without the chain of thought, which the verifiable reward uses for grounding.

In [ ]:
from datasets import load_dataset

from model_io import SYSTEM_PROMPT

full = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train")
RL_ROWS = range(1000, 1256)
EVAL_ROWS = range(2000, 2100)


def to_record(row, idx):
    return {
        "id": f"mo1-{idx}",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["Question"]},
            {"role": "assistant", "content": f"{row['Complex_CoT']}\n\n{row['Response']}"},
        ],
        "reference_answer": row["Response"],
    }


rl_records = [to_record(full[i], i) for i in RL_ROWS]
eval_records = [to_record(full[i], i) for i in EVAL_ROWS]

rl_data_s3 = f"{s3_root}/datasets/rl-alignment/rl"
eval_data_s3 = f"{s3_root}/datasets/rl-alignment/eval"
for records, local_dir, s3_uri in [(rl_records, "./data_rl/rl", rl_data_s3), (eval_records, "./data_rl/eval", eval_data_s3)]:
    os.makedirs(local_dir, exist_ok=True)
    with open(f"{local_dir}/dataset.json", "w", encoding="utf-8") as f:
        json.dump(records, f)
    key = s3_uri.replace(f"s3://{bucket_name}/", "") + "/dataset.json"
    s3_client.upload_file(f"{local_dir}/dataset.json", bucket_name, key)
    print(f"{len(records):4d} records -> {s3_uri}/dataset.json")

### 0.4 Fine-tune Qwen3-0.6B with QLoRA

The RL pass starts from a QLoRA fine-tuned model, which section 4 compares with the base model and the RL-tuned model.

* `START_FROM_BASE = False` (the default) fine-tunes the model first. The cell reuses `SFT_MODEL_S3_URI` if you set it, or a job this lab trained earlier with the same number of examples (`train-Qwen3-0-6B-sft200-rl-lab`). Otherwise the next cell trains one, which takes about 10 minutes.
* `START_FROM_BASE = True` skips fine-tuning and runs RL on the base model. `Qwen/Qwen3-0.6B` is an instruction-tuned chat model that already writes step-by-step answers, so RL can start from it directly. The comparison in sections 5 and 6 is then baseline against RL-tuned.

In [ ]:
from utils import get_last_job_name

START_FROM_BASE = False  # True: skip fine-tuning and run RL directly on the base model
SFT_TRAIN_SAMPLES = 200  # QLoRA training examples; a further 10% is held out for validation

SFT_MODEL_S3_URI = None  # for example "s3://<bucket>/<prefix>/<job>/output/model.tar.gz"
SFT_LAB_JOB_PREFIX = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft{SFT_TRAIN_SAMPLES}-rl-lab"


def latest_model_artifact(job_prefix):
    try:
        job_name = get_last_job_name(job_prefix)
    except ValueError:
        return None
    return sm_client.describe_training_job(TrainingJobName=job_name)["ModelArtifacts"]["S3ModelArtifacts"]


if START_FROM_BASE:
    print(f"RL starts directly from the base model {model_id}.")
else:
    if SFT_MODEL_S3_URI is None:
        SFT_MODEL_S3_URI = latest_model_artifact(SFT_LAB_JOB_PREFIX)
        if SFT_MODEL_S3_URI:
            print(f"Reusing the fine-tuned model from the '{SFT_LAB_JOB_PREFIX}' job.")
    print(f"SFT model: {SFT_MODEL_S3_URI or 'not found, the next cell trains one'}")

With `START_FROM_BASE = False` and no fine-tuned model yet, the next cell fine-tunes Qwen3-0.6B. It uses the first 220 rows of the dataset, 200 for training and 20 for validation, the same system prompt, `scripts_rl/train.py` (a copy of the 02.01 script), the same QLoRA hyperparameters as 02.01, and merged weights. The cell does nothing otherwise.

In [ ]:
if START_FROM_BASE:
    print("START_FROM_BASE is True, skipping SFT training.")
elif SFT_MODEL_S3_URI is None:
    from sagemaker.core.shapes import OutputDataConfig
    from sagemaker.core.training.configs import Compute, SourceCode
    from sagemaker.train.model_trainer import InputData, ModelTrainer, StoppingCondition, Torchrun

    # SFT data: the first rows of the dataset, with only the "messages" column that train.py expects.
    n_val = SFT_TRAIN_SAMPLES // 10
    sft_rows = [{"messages": to_record(full[i], i)["messages"]} for i in range(SFT_TRAIN_SAMPLES + n_val)]
    assert SFT_TRAIN_SAMPLES + n_val <= RL_ROWS.start, "SFT rows must not overlap the RL prompts"
    sft_splits = {"train": sft_rows[n_val:], "test": sft_rows[:n_val]}
    sft_data_s3 = f"{s3_root}/datasets/rl-alignment/sft"
    for split, rows in sft_splits.items():
        key = f"{sft_data_s3}/{split}/dataset.json".replace(f"s3://{bucket_name}/", "")
        s3_client.put_object(Bucket=bucket_name, Key=key, Body=json.dumps(rows).encode("utf-8"))

    sft_args_yaml = f"""model_id: "{model_id}"
mlflow_uri: "{mlflow_tracking_server_uri}"
mlflow_experiment_name: "{mlflow_experiment_name}"
output_dir: "/opt/ml/model"
train_dataset_path: "/opt/ml/input/data/train/"
test_dataset_path: "/opt/ml/input/data/test/"
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1
learning_rate: 2e-4
num_train_epochs: 1
per_device_train_batch_size: 2
per_device_eval_batch_size: 1
gradient_accumulation_steps: 2
gradient_checkpointing: true
fp16: true
bf16: false
tf32: false
merge_weights: true
"""
    sft_config_s3 = f"{s3_root}/datasets/rl-alignment/sft-config"
    s3_client.put_object(Bucket=bucket_name, Key=f"{sft_config_s3}/args.yaml".replace(f"s3://{bucket_name}/", ""),
                         Body=sft_args_yaml.encode("utf-8"))

    sft_trainer = ModelTrainer(
        training_image=TRAINING_IMAGE,
        role=role,
        source_code=SourceCode(source_dir="./scripts_rl", requirements="requirements.txt", entry_script="train.py"),
        base_job_name=SFT_LAB_JOB_PREFIX,
        compute=Compute(instance_type="ml.g5.2xlarge", instance_count=1, volume_size_in_gb=50),
        distributed=Torchrun(),
        stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
        hyperparameters={"config": "/opt/ml/input/data/config/args.yaml"},
        output_data_config=OutputDataConfig(s3_output_path=f"{s3_root}/{SFT_LAB_JOB_PREFIX}"),
        # scripts_rl/requirements.txt pins bitsandbytes 0.45.5, which has a build for the image's CUDA 12.8
        environment={"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"},
    )
    sft_trainer.train(input_data_config=[
        InputData(channel_name="train", data_source=f"{sft_data_s3}/train"),
        InputData(channel_name="test", data_source=f"{sft_data_s3}/test"),
        InputData(channel_name="config", data_source=sft_config_s3),
    ], wait=True)
    sft_job_name = sft_trainer._latest_training_job.training_job_name
    SFT_MODEL_S3_URI = sm_client.describe_training_job(TrainingJobName=sft_job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"SFT model ({SFT_TRAIN_SAMPLES} training examples): {SFT_MODEL_S3_URI}")
else:
    print("Fine-tuned model available, skipping SFT training.")

---
## 1. Define a reward signal that captures a quality dimension the benchmark cannot score

The reward lives in `scripts_rl/rewards.py` and supports both options:

**Option B, RLVR (verifiable reward).** These are deterministic checks that anyone can re-run:
* *Grounding*: the share of the reference answer's key medical terms that appear in the output.
* *Structure*: step-by-step reasoning followed by a distinct final answer, which is what the system prompt asks for.
* *Penalties*: the score is multiplied down for answers over 450 words and for looping, repetitive text. Both are common ways for a policy to exploit a reward.

**Option A, RLAIF (AI feedback).** An Amazon Bedrock judge grades each answer against the reference on three criteria from 1 to 5: correctness of the conclusion (weight 0.5), safety (0.3) and reasoning quality (0.2). The rubric tells the judge not to reward length.

**Hybrid (the default here)** blends them as `0.7 * judge + 0.3 * rlvr`. Each signal covers a weakness of the other: the verifier is cheap and stable but lexical, and the judge understands meaning but is noisy and can be flattered.

The training reward uses Amazon Nova Pro as the judge. Section 4 grades with Claude Sonnet 4.5, a different model family, so that a policy that learned the quirks of its training judge cannot also grade itself.

### 1.1 Compare ROUGE with the verifiable reward

The cell scores three answers to the same held-out question without calling any AWS service. The padded answer copies the reference and repeats it. ROUGE rates it well because it overlaps with the reference, while the verifiable reward penalizes the repetition and the length.

In [ ]:
from rouge_score import rouge_scorer

from model_io import split_record

prompt_msgs, question, reference, reference_answer = split_record(eval_records[0])
reward_cfg_rlvr = rewards.RewardConfig(mode="rlvr")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

candidates = {
    "reference answer": reference,
    "padded copy": " ".join([reference] * 3),
    "bare conclusion": reference_answer.split(".")[0] + ".",
}
rows = []
for name, text in candidates.items():
    detail = rewards.verifiable_reward(text, reference_answer, reward_cfg_rlvr)
    rows.append({"answer": name, "rougeL": scorer.score(reference, text)["rougeL"].fmeasure, **detail})

print(question[:300], "\n")
pd.DataFrame(rows).set_index("answer").round(3)

### 1.2 Check access to the judge models

Check that both judge models are reachable before you launch a job that depends on them. This makes one Bedrock call per judge.

In [ ]:
for label in ("train_judge", "eval_judge"):
    judge = rewards.BedrockJudge(judges[label], region)
    result = judge.score(question, reference, reference)
    if result is None:
        raise RuntimeError(f"{judges[label]} did not return a usable score. Check Bedrock model access and IAM.")
    print(f"{label:12s} {judges[label]}: score={result['score']:.2f} {result}")

### 1.3 Set the reward configuration

This is the configuration the RL job receives in its `REWARD_CONFIG` environment variable. Change `mode` to `"rlvr"` to run without Bedrock, or to `"rlaif"` to use the judge alone.

In [ ]:
reward_config = rewards.RewardConfig(
    mode="hybrid",
    judge_model_id=judges["train_judge"],
    judge_weight=0.7,
    region=region,
)
print(reward_config.to_json())

---
## 2. Configure the RL training job

`scripts_rl/dpo_train.py` runs three phases in one training job:

1. **On-policy preference building.** The starting model (base or fine-tuned, see section 0.4) samples 4 answers per RL prompt at temperature 0.9. On a multi-GPU instance the prompts are split across all GPUs, with one generation process per GPU. Every answer is scored with the reward from section 1. The best and worst answers of each prompt become a (chosen, rejected) pair when their reward gap is at least 0.05.
2. **DPO with QLoRA.** Direct Preference Optimization trains a LoRA adapter on those pairs and merges it into the starting weights, so the output deploys like any Qwen3-0.6B model.
3. **Evaluation.** The job scores the base model, the fine-tuned model if there is one, and the RL-tuned model on the held-out set, which section 4 explains. The models are already on the instance's disk, so this skips a second job with its own start-up and downloads.

DPO optimizes the same objective as PPO-based RLHF, `max E[r(x, y)] - beta * KL(pi || pi_ref)`, without training a separate reward model or running a PPO rollout loop. The reference policy `pi_ref` is the frozen starting model. With a PEFT adapter, TRL uses the same network with the adapter switched off as the reference, so the job holds only one copy of the weights.

### 2.1 Set the hyperparameters and the KL penalty

**The KL penalty is `beta`.** It is the main defense against reward hacking:

| `beta` | Effect |
|---|---|
| 0.5 | Stays very close to the starting model. It is safe, but the gains are small. |
| **0.1** | The standard DPO setting and a good starting point. |
| 0.01 | Follows the reward aggressively, which risks drift, verbosity and judge exploitation. |

Four other controls limit reward hacking. First, the reward already penalizes length and repetition. Second, a small learning rate and 2 epochs keep the update short. Third, section 4 grades with an independent judge. Fourth, section 6 checks that the ROUGE benchmark did not regress.

In [ ]:
hyperparameters = {
    # DPO objective and KL penalty
    "beta": 0.1,
    "loss_type": "sigmoid",
    # optimization
    "learning_rate": 2e-5,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "max_length": 1536,
    "max_prompt_length": 512,
    "bf16": True,
    "gradient_checkpointing": True,
    "report_to": "mlflow" if mlflow_tracking_server_uri else "none",
    "output_dir": "/opt/ml/model",
    # preference building
    "sft_model_path": model_id if START_FROM_BASE else "/opt/ml/input/data/model",  # starting policy
    "max_prompts": 256,
    "num_candidates": 4,
    "min_reward_margin": 0.05,
    "gen_max_new_tokens": 512,
    "gen_batch_size": 32,
    "judge_workers": 16,
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    # evaluation inside the same job (section 4)
    "run_final_eval": True,
    "final_eval_base_model_id": model_id,
    "final_eval_judge_model_id": judges["eval_judge"],
    "final_eval_bedrock_region": region,
    "final_eval_max_samples": 100,
    "final_eval_batch_size": 32,
}

hyperparameters

### 2.2 Set the instance type and training container

The RL job uses the same SOCI-indexed container as the fine-tuning job. `scripts_rl/requirements.txt` pins bitsandbytes 0.45.5, which includes a CUDA 12.8 build for this image.

In [ ]:
# The RL job runs on one GPU. ml.g5.12xlarge is used because it usually has more available capacity than
# ml.g5.2xlarge; switch back to "ml.g5.2xlarge" if that size starts faster in your Region.
instance_type = "ml.g5.12xlarge"

instance_type

In [ ]:
image_uri = TRAINING_IMAGE  # SOCI-indexed PyTorch DLC from section 0.1

image_uri

### 2.3 Define the ModelTrainer and the input channels

The job receives the RL prompts in the `prompts` channel and the held-out evaluation set in the `eval` channel, both from section 0.3. With `START_FROM_BASE = False` it also receives the fine-tuned `model.tar.gz` in the `model` channel. When RL starts from the base model, the job downloads it from the Hugging Face Hub instead. The reward configuration and the MLflow settings (section 0.2) arrive as environment variables.

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer, InputData, StoppingCondition
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.core.shapes import OutputDataConfig

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts_rl",
    requirements="requirements.txt",
    entry_script="dpo_train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=100,
)

# define Training Job Name
rl_job_prefix = f"train-{model_id.split('/')[-1].replace('.', '-')}-rl-dpo"

# define OutputDataConfig path
if default_prefix:
    rl_output_path = f"s3://{bucket_name}/{default_prefix}/{rl_job_prefix}"
else:
    rl_output_path = f"s3://{bucket_name}/{rl_job_prefix}"

environment = {
    "REWARD_CONFIG": reward_config.to_json(),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
}
if mlflow_tracking_server_uri:
    import time

    environment.update({
        "MLFLOW_TRACKING_URI": mlflow_tracking_server_uri,
        "MLFLOW_EXPERIMENT_NAME": mlflow_experiment_name,
        "MLFLOW_RUN_NAME": f"rl-dpo-{time.strftime('%Y-%m-%d-%H-%M')}",
        "MLFLOW_TAG_START_FROM_BASE": str(START_FROM_BASE),
    })

# Define the ModelTrainer
rl_trainer = ModelTrainer(
    training_image=image_uri,
    role=role,
    source_code=source_code,
    base_job_name=rl_job_prefix,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=2 * 3600),
    hyperparameters=hyperparameters,
    output_data_config=OutputDataConfig(s3_output_path=rl_output_path),
    environment=environment,
)

In [ ]:
# Pass the input data
prompts_input = InputData(
    channel_name="prompts",
    data_source=rl_data_s3,  # RL prompts from section 0.3
)

eval_input = InputData(
    channel_name="eval",
    data_source=eval_data_s3,  # held-out evaluation set from section 0.3
)

data = [prompts_input, eval_input]

if not START_FROM_BASE:
    data.append(InputData(
        channel_name="model",
        data_source=SFT_MODEL_S3_URI,  # fine-tuned model.tar.gz from section 0.4
    ))

# Check input channels configured
data

---
## 3. Launch the RL training job

The job takes about 20 minutes on `ml.g5.12xlarge`, including the evaluation phase. About 3 minutes go to start-up and the package install, about 3 minutes to sampling 1,024 candidate answers on the instance's 4 GPUs, about 1 minute to the judge calls, about 4 minutes to DPO and the merge, and the rest to evaluating the two models.

### 3.1 Start the training job

In [ ]:
# starting the train job with the fine-tuned model and the RL prompts as input
rl_trainer.train(input_data_config=data, wait=True)

### 3.2 Locate the RL-tuned model

In [ ]:
# the job that section 3.1 just ran (SageMaker Search can lag a few minutes behind job completion)
rl_job_name = rl_trainer._latest_training_job.training_job_name
rl_job = sm_client.describe_training_job(TrainingJobName=rl_job_name)

RL_MODEL_S3_URI = rl_job["ModelArtifacts"]["S3ModelArtifacts"]
rl_output_tar = f"{rl_job['OutputDataConfig']['S3OutputPath'].rstrip('/')}/{rl_job_name}/output/output.tar.gz"

print(f"RL job   : {rl_job_name} ({rl_job['TrainingJobStatus']})")
print(f"RL model : {RL_MODEL_S3_URI}")
print(f"Outputs  : {rl_output_tar}")

### 3.3 Inspect the preferences and the DPO training curves

`preference_stats.json` shows how far apart the chosen and rejected answers were. The DPO log shows `rewards/margins`, which is how strongly the policy now prefers chosen over rejected, and `rewards/accuracies`.

DPO's implicit reward is `beta * (log pi(y|x) - log pi_ref(y|x))`, so dividing `rewards/chosen` by `beta` gives the average log-probability ratio against the starting model. It is a direct view of how far the KL penalty let the policy move. If this ratio keeps growing while the margins flatten, lower the learning rate or raise `beta`.

In [ ]:
import tarfile

import matplotlib.pyplot as plt

os.makedirs("./rl_job_output", exist_ok=True)
bucket, key = rl_output_tar.replace("s3://", "").split("/", 1)
s3_client.download_file(bucket, key, "./rl_job_output/output.tar.gz")
with tarfile.open("./rl_job_output/output.tar.gz") as tar:
    tar.extractall("./rl_job_output")
with open("./rl_job_output/preference_stats.json") as f:
    print(json.dumps(json.load(f), indent=2))
with open("./rl_job_output/dpo_log_history.json") as f:
    history = [h for h in json.load(f) if "rewards/margins" in h]

beta = hyperparameters["beta"]
steps = [h["step"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(steps, [h["loss"] for h in history]); axes[0].set_title("DPO loss")
axes[1].plot(steps, [h["rewards/margins"] for h in history], label="margin")
axes[1].plot(steps, [h["rewards/accuracies"] for h in history], label="accuracy"); axes[1].legend()
axes[1].set_title("Preference margin and accuracy")
axes[2].plot(steps, [h["rewards/chosen"] / beta for h in history], label="chosen")
axes[2].plot(steps, [h["rewards/rejected"] / beta for h in history], label="rejected"); axes[2].legend()
axes[2].set_title("log pi/pi_ref (drift from the starting model)")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout()
plt.show()

---
## 4. Re-evaluate against the same baseline benchmark

The last phase of the RL job evaluated the models on the 100 held-out prompts, one after another on the same GPU:

* **base**: `Qwen/Qwen3-0.6B`, downloaded from the Hugging Face Hub
* **sft**: the fine-tuned model from section 0.4, only with `START_FROM_BASE = False`
* **rl**: the RL-tuned model from section 3

Every model gets the same system prompt and the same decoding settings as the task 03 benchmark (temperature 0.6, top_p 0.9, 512 new tokens). `scripts_rl/evaluate.py` scores each answer with the benchmark, the verifiable reward and the independent eval judge, which section 4.1 explains. The job also ran the paired significance tests that section 6 explains.

The cell below copies the evaluation results from the job output to their own S3 prefix.

### 4.1 How the evaluation works and what each metric means

**How it runs.** Every model answers the same 100 held-out questions with the same decoding settings. Each answer gets several scores, and section 6 compares two models question by question. Because the questions are shared, the comparison is *paired*: some questions are hard for every model, and pairing removes that question-to-question variation, so smaller differences become detectable.

**Per-answer scores** (0 to 1, higher is better, except `words`):

| Metric | What it measures | How it is computed | How to read it |
|---|---|---|---|
| `rouge1`, `rouge2`, `rougeL` | Word overlap with the reference answer: single words, word pairs, and the longest shared word sequence. `rougeL` is the task 03 benchmark. | ROUGE F1 between the answer and the reference (reasoning plus final answer) | Rewards *sounding like* the reference, not being correct. Values of 0.15 to 0.20 are normal for long free-form answers. |
| `rlvr` | Rule-based quality | 0.7 x the share of the reference answer's key medical terms that appear, plus 0.3 x structure (step-by-step reasoning, then a separate final answer), multiplied down for answers over 450 words and for repetitive loops | Cheap, deterministic and auditable, but lexical: mentioning the right terms is not the same as reaching the right conclusion. |
| `judge` | Clinical quality | An independent Amazon Bedrock judge (Claude Sonnet 4.5) grades correctness (weight 0.5), safety (0.3) and reasoning (0.2) from 1 to 5 against the reference; the weighted grade is mapped to 0 to 1 | The closest to "is this a good medical answer". A value of 0.25 means average grades of about 2 out of 5. The per-answer grades and a one-sentence rationale are in `per_sample_<model>.jsonl`. |
| `reward` | The target the RL pass optimizes | 0.7 x `judge` + 0.3 x `rlvr` (the section 1 composite, recomputed with the independent judge) | The main metric for judging the RL step. |
| `words` | Answer length | Word count | Not a quality score. A reward gain that comes with much longer answers is a warning sign of reward hacking. |

In [ ]:
local_eval = "./rl_job_output/eval"
RL_EVAL_S3_URI = f"{s3_root}/rl-alignment/evaluation/{rl_job_name}"

eval_files = sorted(os.listdir(local_eval))
for name in eval_files:
    key = f"{RL_EVAL_S3_URI}/{name}".replace(f"s3://{bucket_name}/", "")
    s3_client.upload_file(os.path.join(local_eval, name), bucket_name, key)

print(f"Evaluation results: {RL_EVAL_S3_URI}")
print("\n".join(f"  {name}" for name in eval_files))

---
## 5. Compare metrics: baseline, fine-tuned (optional), RL-tuned

The first three columns are the benchmark from task 03. `rlvr`, `judge` and `reward` measure the dimension the RL pass targeted. `words` is the mean answer length. A large jump in length alongside a reward gain is the classic sign of reward hacking.

In [ ]:
with open(f"{local_eval}/evaluation.json") as f:
    evaluation = json.load(f)

metric_cols = ["rouge1", "rouge2", "rougeL", "rlvr", "judge", "reward", "words"]
models_evaluated = [m for m in ("base", "sft", "rl") if m in evaluation["summary"]]
summary_df = pd.DataFrame(evaluation["summary"]).T.loc[models_evaluated, metric_cols]
display(summary_df.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
summary_df[metric_cols[:-1]].T.plot.bar(ax=ax)
ax.set_title(f"{', '.join(models_evaluated)} on the same {len(eval_records)} held-out prompts")
ax.set_ylabel("mean score (0 to 1)")
ax.legend(title="model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Analyze whether the gain is statistically significant

For a comparison such as `sft -> rl`, every question gives one difference: the candidate's score minus the baseline's score. That is 100 differences per metric. `scripts_rl/rl_stats.py` turns them into the columns of the tables below:

| Column | Meaning | How to read it |
|---|---|---|
| `delta` | Mean of the 100 differences | Positive means the candidate is better on average. |
| `95% CI` | Bootstrap confidence interval: resample the differences with replacement 10,000 times and keep the middle 95% of the resampled means | The range where the true improvement plausibly lies. If it contains 0, "no improvement" cannot be ruled out. |
| `p (Holm)` | Permutation-test p-value, corrected with Holm-Bonferroni (explained below) | The chance of seeing a difference this large if the two models were equally good. Below 0.05 means the difference is unlikely to be luck. |
| `d_z` | Effect size: mean difference divided by the standard deviation of the differences | Size of the change relative to its noise, independent of sample size. About 0.2 is small, 0.5 medium, 0.8 large. |
| `significant gain` | `True` only if the Holm p-value is at most 0.05 **and** the whole CI is above 0 | A gain that is both unlikely to be luck and positive across the plausible range. |

**The permutation test.** If the two models were equally good, each difference would be as likely to be positive as negative. The test flips the sign of every difference at random 10,000 times and counts how often the shuffled mean is at least as far from zero as the real one. That share is the p-value. It makes no assumption that scores are normally distributed, which matters for bounded, skewed scores such as ROUGE.

**The Holm-Bonferroni correction.** Each comparison tests six metrics at once. At a 5% false-positive rate per test, the chance that at least one of six shows a "significant" result by luck alone is about 26%. Holm corrects for this:

1. Sort the six p-values from smallest to largest.
2. Multiply the smallest by 6, the next by 5, and so on down to 1.
3. Make the results non-decreasing in that order and cap them at 1.

For example, raw p-values of 0.01, 0.03 and 0.04 for three metrics become 0.03, 0.06 and 0.06: only the first stays significant. The correction keeps the false-positive rate at 5% for the whole table.

**How small a gain can this detect?** With 100 questions and this much answer-to-answer noise, the smallest gain the test detects reliably is about 0.025 to 0.03 on a 0 to 1 score. A real but smaller gain often shows up as "not significant". Section 6.4 lists this and the other limits.

### 6.1 Significance tables

In [ ]:
def significance_table(comparison_name):
    rows = []
    for metric, s in evaluation["comparisons"][comparison_name].items():
        rows.append({
            "metric": metric, "before": s["baseline_mean"], "after": s["candidate_mean"], "delta": s["delta"],
            "95% CI": f"[{s['ci_low']:+.3f}, {s['ci_high']:+.3f}]", "p (Holm)": s["p_value_holm"],
            "d_z": s["effect_size_dz"], "significant gain": s["significant"],
        })
    return pd.DataFrame(rows).set_index("metric")


for name in evaluation["comparisons"]:
    print(f"\n{name.replace('_vs_', ' -> ')}")
    display(significance_table(name).round(4))

### 6.2 Verdict on the RL step

The verdict applies three checks to the RL step: `base -> rl` when RL started from the base model, or `sft -> rl` after fine-tuning. First, the targeted reward must improve significantly. Second, the benchmark must not regress by more than 0.02 ROUGE-L. Third, answers must not grow by more than 25%, which would suggest the policy gamed the reward with length. The cell recomputes the comparison locally from the per-sample files, which is how you audit the job's numbers.

In [ ]:
start = "sft" if "sft" in evaluation["summary"] else "base"  # the model the RL pass started from
per_sample = {}
for name in (start, "rl"):
    with open(f"{local_eval}/per_sample_{name}.jsonl") as f:
        per_sample[name] = [json.loads(line) for line in f]

local = rl_stats.compare_models(per_sample[start], per_sample["rl"], ["reward", "rougeL", "words"])
reward_ok = local["reward"]["significant"]
benchmark_ok = local["rougeL"]["delta"] >= -0.02
length_ok = local["words"]["candidate_mean"] <= 1.25 * local["words"]["baseline_mean"]

print(f"RL step: {start} -> rl")
print(f"reward gain significant:        {reward_ok}  (delta {local['reward']['delta']:+.3f}, p_holm {local['reward']['p_value_holm']:.4f})")
print(f"benchmark did not regress:      {benchmark_ok}  (ROUGE-L delta {local['rougeL']['delta']:+.3f})")
print(f"no length-driven reward gain:   {length_ok}  ({local['words']['baseline_mean']:.0f} -> {local['words']['candidate_mean']:.0f} words)")
print("\nVERDICT:", "the RL gain is real on the targeted dimension." if reward_ok and benchmark_ok and length_ok
      else "the evidence does not support an RL improvement yet.")

# skip prompts where a judge call returned no score, so they are not shown as regressions
scored = [p for p in zip(per_sample[start], per_sample["rl"]) if p[0]["reward"] is not None and p[1]["reward"] is not None]
worst = sorted(scored, key=lambda p: p[1]["reward"] - p[0]["reward"])[:2]
for before, after in worst:
    print(f"\n--- largest regression, prompt {before['id']}: reward {before['reward']} -> {after['reward']}")
    print(f"{start.upper()}:", before["prediction"][:500])
    print("RL: ", after["prediction"][:500])

### 6.3 Review the runs in MLflow

The RL job logged the same numbers you see above to MLflow, together with the preference statistics and the DPO training curves. The cell lists this experiment's runs and creates a link that opens the MLflow UI for 30 minutes. Use the UI to compare runs side by side, for example two values of `beta` or a run with and without fine-tuning.

In [ ]:
if mlflow_tracking_server_uri:
    import mlflow

    mlflow.set_tracking_uri(mlflow_tracking_server_uri)
    runs = mlflow.search_runs(experiment_names=[mlflow_experiment_name], order_by=["start_time DESC"], max_results=10)
    cols = [col for col in ("tags.mlflow.runName", "start_time", "metrics.gate.primary_delta", "metrics.gate.primary_p_value",
                            "metrics.eval.rl.reward", "metrics.preferences.pairs", "params.dpo.beta") if col in runs.columns]
    display(runs[cols])

    url = sm_client.create_presigned_mlflow_tracking_server_url(
        TrackingServerName=mlflow_tracking_server_name, ExpiresInSeconds=300, SessionExpirationDurationInSeconds=1800
    )["AuthorizedUrl"]
    print(f"MLflow UI: {url}")
else:
    print("No MLflow tracking server: see section 0.2.")

### 6.4 Scope and limits of this measurement

* The test uses 100 prompts with one sampled answer per prompt, from one training seed. A difference of about 0.025 to 0.03 on a 0 to 1 score is near the smallest effect this setup can detect, so a real but smaller gain can show as not significant. Raise `EVAL_ROWS` and `final_eval_max_samples` for a more sensitive test.
* The RL pass learns from the preference pairs built from 256 prompts (`preference_stats.json` in section 3.3 shows how many passed the margin filter). More prompts give a stronger signal at the cost of a longer job.
* A 0.6B model reasons much less reliably than the 4B model from 02.01, so expect lower absolute scores on every metric.
* The judge scores are themselves noisy. The eval judge is independent of the training judge, but it is still an LLM, so read the per-sample rationales in `per_sample_rl.jsonl` before you trust a small gain.
* ROUGE did not drop, but that does not show that clinical accuracy improved. Human review of a sample of answers remains the final check before production.

---
## 7. Deploy the RL-tuned model

Deploy the RL-tuned model on an Amazon SageMaker AI real-time endpoint, with the same DJL Large Model Inference (LMI) container and vLLM settings as Lab 02.01. `task_04_responsible_ai/04.01_bedrock_guardrails_apply_guardrail_api.ipynb` uses this endpoint to apply Amazon Bedrock Guardrails to the RL-tuned model: the endpoint name is stored as `TUNED_ENDPOINT_NAME`, the variable 04.01 reads. 04.01 sends OpenAI-style `messages`, for which the endpoint applies the Qwen3 chat template itself, so Qwen3-0.6B starts its answers there with a `<think>` reasoning block.

The settings differ from 02.01 in one place: there is no FP8 quantization, because a 0.6B model fits easily on one A10G GPU in bf16.

### 7.1 Inference configurations

In [ ]:
instance_count = 1
endpoint_instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

inference_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.33.0-lmi15.0.0-cu128"
print(f"using image to host: {inference_image_uri}")

In [ ]:
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

RL_ENDPOINT_NAME = name_from_base(f"{model_id.split('/')[-1].replace('.', '-')}-rl")

core_model = Model.create(
    model_name=RL_ENDPOINT_NAME,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=inference_image_uri,
        model_data_url=RL_MODEL_S3_URI,  # merged RL-tuned model from section 3
        environment={
            'HF_MODEL_ID': "/opt/ml/model",
            'OPTION_TRUST_REMOTE_CODE': 'true',
            'OPTION_ROLLING_BATCH': "vllm",
            'OPTION_DTYPE': 'bf16',
            'OPTION_TENSOR_PARALLEL_DEGREE': 'max',
            'OPTION_MAX_ROLLING_BATCH_SIZE': '32',
            'OPTION_MODEL_LOADING_TIMEOUT': '3600',
            'OPTION_MAX_MODEL_LEN': '4096'
        }
    ),
)

In [ ]:
from sagemaker.core.helper.session_helper import _wait_until, _deploy_done

EndpointConfig.create(
    endpoint_config_name=RL_ENDPOINT_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=RL_ENDPOINT_NAME,
            initial_instance_count=instance_count,
            instance_type=endpoint_instance_type,
            container_startup_health_check_timeout_in_seconds=health_check_timeout,
            model_data_download_timeout_in_seconds=3600,
        )
    ],
)

core_endpoint = Endpoint.create(
    endpoint_name=RL_ENDPOINT_NAME,
    endpoint_config_name=RL_ENDPOINT_NAME,
)

_wait_until(lambda: _deploy_done(sagemaker_session.sagemaker_client, RL_ENDPOINT_NAME), poll=30)

core_endpoint = Endpoint.get(endpoint_name=RL_ENDPOINT_NAME)
print(f"Endpoint status: {core_endpoint.endpoint_status}")

### 7.2 Predict

The request sends the prompt already formatted with the Qwen3 chat template, with an empty thinking block. This is the exact format the RL job trained and evaluated on. If the endpoint applied the template itself, Qwen3-0.6B would open a `<think>` section first.

The raw-prompt route of the LMI container returns the generated text as byte-level BPE tokens, so `decode_generated_text` converts it back to plain text.

In [ ]:
def decode_generated_text(text):
    """Decode the byte-level BPE tokens that the LMI raw-prompt route returns ("\u0120" for a space, "\u010a"
    for a newline) back into plain text, and drop the end-of-turn token."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + list(range(ord("\xae"), ord("\xff") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    byte_of = {chr(c): b for b, c in zip(bs, cs)}
    if any(ch in text for ch in ("\u0120", "\u010a")):
        text = bytes(byte_of.get(ch, 0) for ch in text if ch in byte_of).decode("utf-8", errors="replace")
    return text.split("<|im_end|>")[0].strip()


def build_prompt(system_prompt, user_prompt):
    """Qwen3 chat format with thinking turned off (an empty <think> block), as used in training."""
    return (f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
            "<|im_start|>assistant\n<think>\n\n</think>\n\n")


USER_PROMPT = "A 3-week-old child has been diagnosed with late onset perinatal meningitis, and the CSF culture shows gram-positive bacilli. What characteristic of this bacterium can specifically differentiate it from other bacterial agents?"

payload = json.dumps({
    "inputs": build_prompt(SYSTEM_PROMPT, USER_PROMPT),
    "parameters": {
        "temperature": 0.6,
        "top_p": 0.9,
        "max_new_tokens": 512
    }
})

response = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))
result = result[0] if isinstance(result, list) else result
print(decode_generated_text(result["generated_text"]))

---
## Store variables for downstream notebooks

Saves these values to the IPython store so `04.01_bedrock_guardrails_apply_guardrail_api.ipynb` and `05.02_rl_improvement_pipeline.ipynb` can load them with `%store -r`.

| Variable | Description |
|---|---|
| `SFT_MODEL_S3_URI` | Fine-tuned model used as the RL starting point, or `None` when RL started from the base model (section 0.4) |
| `RL_MODEL_S3_URI` | RL-tuned merged model (section 3) |
| `RL_EVAL_S3_URI` | Evaluation outputs: per-sample results, summary and significance tests (section 4) |
| `RL_DATASET_S3_PREFIX` | RL prompts and held-out evaluation set (section 0.3) |
| `RL_REWARD_CONFIG` | Reward configuration JSON (section 1.3) |
| `RL_ENDPOINT_NAME` | Real-time endpoint serving the RL-tuned model (section 7) |
| `TUNED_ENDPOINT_NAME` | Same endpoint, under the name `04.01_bedrock_guardrails_apply_guardrail_api.ipynb` reads. This replaces the 02.01 endpoint name in the store. |

In [ ]:
RL_DATASET_S3_PREFIX = f"{s3_root}/datasets/rl-alignment"
RL_REWARD_CONFIG = reward_config.to_json()

%store SFT_MODEL_S3_URI
%store RL_MODEL_S3_URI
%store RL_EVAL_S3_URI
%store RL_DATASET_S3_PREFIX
%store RL_REWARD_CONFIG
%store RL_ENDPOINT_NAME

TUNED_ENDPOINT_NAME = RL_ENDPOINT_NAME  # read by 04.01
%store TUNED_ENDPOINT_NAME

---
## Cleanup

The training jobs stop by themselves. The endpoint from section 7 keeps running, and is billed, because `04.01_bedrock_guardrails_apply_guardrail_api.ipynb` uses it; that notebook's clean-up deletes the endpoint and its endpoint configuration. Set `cleanup = True` to delete the endpoint here instead, together with the model artifacts and evaluation results in S3.

In [ ]:
cleanup = False

In [ ]:
if cleanup:
    for delete, kwargs in [(sm_client.delete_endpoint, {"EndpointName": RL_ENDPOINT_NAME}),
                           (sm_client.delete_endpoint_config, {"EndpointConfigName": RL_ENDPOINT_NAME}),
                           (sm_client.delete_model, {"ModelName": RL_ENDPOINT_NAME})]:
        try:
            delete(**kwargs)
            print(f"Deleted {list(kwargs.values())[0]}")
        except ClientError as err:
            print(f"Could not delete {list(kwargs.values())[0]}: {err}")

    s3_resource = boto3.resource("s3")
    for uri in (rl_output_path, f"{s3_root}/rl-alignment", f"{s3_root}/datasets/rl-alignment",
                f"{s3_root}/{SFT_LAB_JOB_PREFIX}"):
        bucket, prefix = uri.replace("s3://", "").split("/", 1)
        deleted = s3_resource.Bucket(bucket).objects.filter(Prefix=prefix + "/").delete()
        print(f"Deleted {sum(len(d.get('Deleted', [])) for d in deleted)} objects under {uri}")